# Notebook 3 -- Predicting the LLMs' votes from dialogue features

**Task.** For each (LLM, game): which option does the LLM pick? The ballot is
the roster **plus "No Werewolf"**, so the task is a discrete choice over that
set -- every alternative gets a score, the argmax is the prediction, and the
metric is **top-1 accuracy per game**. This differs from Lai et al.'s pairwise
task (did A vote B, F1 0.33), so numbers are NOT comparable across the two
setups; a framing section near the end reports both readings of the same
predictions and explains why the choice framing is primary here.

**Labels.** Two label sources, never pooled: `stochastic` uses each of the 3
T=1 run votes as its own labeled instance (all instances of a game stay in the
same CV fold), so split games are kept and the reported metric is the expected
probability that a single T=1 sample matches the prediction; `greedy` uses the
single deterministic vote. "No Werewolf" votes are *kept* -- they are a choice
the LLM made, and dropping them would discard 6-25% of the instances
depending on the LLM (stochastic labels: 2B 23%, 31B 12%, 4B 6%) and silently
redefine the task -- and that spread is itself a result, since the smallest
model abstains four times as often as 4B. Only unparseable responses are excluded.

**Feature families** (nested by design):
- **A_pt** -- human-annotated persuasion techniques, speaker perspective:
  per-player counts of Accusation, Defense, Interrogation, Identity
  Declaration, Evidence, Call for Action, plus utterance count.
- **B_directed** -- the DeepSeek-enriched layer: werewolf- and deception-type
  accusations *received*, self-claimed Werewolf, role-claim conflicts,
  self-contradictions.
- **C_combined** -- A + B.

Every family also carries **alternative-specific covariates** for the "No
Werewolf" option (how concentrated the suspicion in that game was): they are
zero on player rows, so they only move the circle alternative's score.

**Feature scaling.** Two variants, chosen by CV: `raw` counts, and
`within_game` counts z-scored across the roster of that game. The metric is a
within-game argmax, so a player's standing *relative to this table* is what can
decide the vote; raw counts also encode how chatty the whole game was.

**Methods.** `logreg` (L2, uses every feature), `gbm` (depth-2 stumps, balanced
sample weights), `lasso` (L1, zeroes out features), and `l1_select` (L1 used as
a selector inside each training fold, then an unpenalised refit on the
survivors). The last two choose their own feature subset.

**Baselines** (also findings): uniform random, most-talkative, most-accused
(werewolf-type), a circle-aware rule (most-accused unless nobody was accused),
and **crowd-modal** -- predict the human village modal target. Crowd-modal is
deliberately a baseline and not a feature: human votes are almost never
declared in the transcript, so the LLM cannot see them; matching this baseline
measures whether LLMs and humans *interpret games similarly*, nothing causal.

**Evaluation.** GroupKFold by game, 5 folds x 3 shuffle seeds. Two layers:
a flat grid over all 40 configurations (exploratory -- its winner is picked on
the folds that report it), and **nested CV**, where an inner 3-fold CV picks
the configuration inside each outer training set. *The nested number is the one
to quote.* Interpretation -- held-out permutation importance -- is reported
only for the configuration nested CV selected most often per LLM, on the
stochastic label source.


## Config and paths

In [1]:
from pathlib import Path
from typing import Optional
from collections import Counter
from itertools import combinations, product
import json
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_score, recall_score)

REPO_NAME = "masters_thesis_sdg"
MODEL_STAGE = "base"
PROMPT_DIR = "prompt_v4"
STOCHASTIC_RUNS = ["run_1", "run_2", "run_3"]
GREEDY_RUN = "greedy_t0"
N_FOLDS = 5           # outer folds
INNER_FOLDS = 3       # inner folds used by nested CV to pick the configuration
CV_SEEDS = [0, 1, 2]
RANDOM_STATE = 42
# Interpretation (importance / selection) is reported for one label source only:
# stochastic has 3x the instances, so held-out drops are far less noisy.
IMPORTANCE_LABEL_SOURCE = "stochastic"

# "No Werewolf" is a real option on the ballot, so it is an alternative in the
# choice set rather than a dropped instance -- see the framing note below.
CIRCLE_OPTION = "__NO_WEREWOLF__"

PT_LABELS = ["Accusation", "Defense", "Interrogation", "Identity Declaration",
             "Evidence", "Call for Action"]

FAMILY_A = [f"pt_{l.lower().replace(' ', '_')}" for l in PT_LABELS] + ["n_utterances"]
FAMILY_B = ["werewolf_count", "deception_count", "claims_werewolf",
            "made_any_claim", "claims_info_role", "n_distinct_roles_claimed_self",
            "is_in_role_conflict", "is_self_contradiction"]
PLAYER_FEATURES = FAMILY_A + FAMILY_B
# Provenance families (did the DeepSeek enrichment add value?)
FAMILIES = {"A_pt": FAMILY_A, "B_directed": FAMILY_B, "C_combined": FAMILY_A + FAMILY_B}
# Mechanism families (the RQ contrast): social-pressure signals vs
# consistency/deduction signals. claims_werewolf (taking a self-declaration
# literally) is grouped with the deductive/content side -- move it if you
# disagree; accusation COUNTS are social salience (the content of an
# accusation may carry evidence, but the count does not).
FAMILY_SOCIAL = FAMILY_A + ["werewolf_count", "deception_count"]
FAMILY_DEDUCTIVE = ["is_in_role_conflict", "is_self_contradiction", "claims_werewolf",
                    "made_any_claim", "claims_info_role", "n_distinct_roles_claimed_self"]
FAMILIES.update({"S_social": FAMILY_SOCIAL, "D_deductive": FAMILY_DEDUCTIVE})

# Alternative-specific covariates for the "No Werewolf" option: zero on every
# player row, so they only move the circle alternative's score. Without them the
# model could never learn WHEN to prefer it -- a feature that is constant within
# a game cannot change a within-game argmax.
CIRCLE_FEATURES = ["is_circle_option", "circ_max_werewolf_count",
                   "circ_mean_werewolf_count", "circ_share_players_accused",
                   "circ_any_claims_werewolf", "circ_any_role_conflict"]
ALL_FEATURES = PLAYER_FEATURES + CIRCLE_FEATURES

# logreg: L2, keeps every feature.  gbm: picks its own splits, so features can
# end up with zero gain.  lasso / l1_select: embedded L1 selection, exact zeros.
# l1_select does the selection inside the training fold and then refits an
# unpenalised model on the survivors -- the "let the model choose the features"
# condition, evaluated honestly on held-out games.
METHODS = ["logreg", "gbm", "lasso", "l1_select"]
# raw counts vs counts z-scored WITHIN each game. The metric is a within-game
# argmax, so "three accusations" only means something relative to the rest of
# this roster; within_game makes the features match the task.
SCALINGS = ["raw", "within_game"]


def find_repo_root(start=None, repo_name=REPO_NAME):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(repo_name)
        current = current.parent


REPO_ROOT = find_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "analysis"
TABLES_REL = Path(MODEL_STAGE) / "voting" / PROMPT_DIR / "vote_stability" / "tables"
ANNOT_ROOT = REPO_ROOT / "data" / "lai2023"
ACC_TARGETS_ROOT = REPO_ROOT / "data" / "processed" / "lai2023" / "accusation_transcripts" / "acc_targets"
IC_FEATURES_CSV = (REPO_ROOT / "data" / "processed" / "lai2023"
                   / "identity_claim_transcripts" / "ic_targets" / "player_conflict_features.csv")
HUMAN_TABLES_DIR = ANALYSIS_ROOT / "human_outcomes" / PROMPT_DIR / "tables"
OUTPUT_DIR = ANALYSIS_ROOT / "cross_model" / "voting" / PROMPT_DIR / "predictive" / "tables"
print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg


## Canonical game keys

Annotations, DeepSeek outputs, and the vote tables spell sessions differently
(`#` vs spaces, case). Everything is joined on a canonical key.


In [2]:
def canonical_session(x):
    x = str(x).strip().replace("#", " ")
    return re.sub(r"\s+", " ", x).lower()


def canonical_game(x):
    m = re.search(r"\d+", str(x))
    return f"game{int(m.group())}" if m else str(x).strip().lower()


def ckey(source, session, game):
    return (str(source).strip(), canonical_session(session), canonical_game(game))


## Labels and rosters (from notebook 1's tables)

One labeled instance per (LLM, game, run) with a named vote. Rosters come from
the game-level table so silent players still appear as candidates.


In [3]:
def short_model_label(name):
    m = re.search(r"(\d+B)", name)
    return m.group(1) if m else name


file_frames, game_frames = [], []
for model_dir in sorted(ANALYSIS_ROOT.iterdir()):
    tables = model_dir / TABLES_REL
    if not (tables / "llm_vote_file_level.csv").exists():
        continue
    label = short_model_label(model_dir.name)
    f = pd.read_csv(tables / "llm_vote_file_level.csv"); f["model"] = label
    g = pd.read_csv(tables / "llm_vote_game_level.csv"); g["model"] = label
    file_frames.append(f); game_frames.append(g)
votes = pd.concat(file_frames, ignore_index=True)
games = pd.concat(game_frames, ignore_index=True)
MODELS = sorted(votes["model"].unique())
print("Models:", MODELS)

roster_by_key = {}
for _, r in games.drop_duplicates(subset=["source", "session_name", "game_key"]).iterrows():
    roster_by_key[ckey(r["source"], r["session_name"], r["game_key"])] = {
        "source": r["source"], "session": r["session_name"], "game": r["game_key"],
        "players": json.loads(r["player_names"]) if isinstance(r["player_names"], str) else r["player_names"],
    }
print("games with rosters:", len(roster_by_key))

# A circle vote ("No Werewolf") is a choice the LLM actually made, so it becomes
# an instance whose target is the CIRCLE_OPTION alternative. Only unparseable
# responses are dropped.
instance_rows, excluded = [], Counter()
for _, r in votes.iterrows():
    label_source = "greedy" if r["run_label"] == GREEDY_RUN else "stochastic"
    if r["run_label"] not in STOCHASTIC_RUNS and r["run_label"] != GREEDY_RUN:
        continue
    if r["status"] not in ("player_vote", "circle_vote"):
        excluded[(label_source, r["status"])] += 1
        continue
    voted = CIRCLE_OPTION if bool(r["is_circle_vote"]) else r["chosen_player_name"]
    if not isinstance(voted, str) or not voted.strip():
        excluded[(label_source, "empty_target")] += 1
        continue
    instance_rows.append({"model": r["model"], "label_source": label_source,
                          "key": ckey(r["source"], r["session_name"], r["game_key"]),
                          "run_label": r["run_label"], "voted": voted,
                          "is_circle": int(voted == CIRCLE_OPTION)})
instances = pd.DataFrame(instance_rows)
print("labeled instances:", instances.groupby(["model", "label_source"]).size().to_dict())
print("excluded (unusable response):", dict(excluded))
print("\nshare of instances that are 'No Werewolf' (now kept, previously dropped):")
display(instances.pivot_table(index="model", columns="label_source",
                              values="is_circle", aggfunc=["mean", "sum"]).round(3))


Models: ['2B', '31B', '4B']
games with rosters: 191


labeled instances: {('2B', 'greedy'): 191, ('2B', 'stochastic'): 568, ('31B', 'greedy'): 191, ('31B', 'stochastic'): 573, ('4B', 'greedy'): 191, ('4B', 'stochastic'): 573}
excluded (unusable response): {('stochastic', 'failed_parse'): 5}

share of instances that are 'No Werewolf' (now kept, previously dropped):


mean               sum           
label_source greedy stochastic greedy stochastic
model                                           
2B            0.251      0.231     48        131
31B           0.126      0.120     24         69
4B            0.094      0.059     18         34

## Family A: persuasion-technique counts (human annotations)

All splits of both sources are pooled -- splits are rebuilt as k-folds below.
The loader reports the annotation-to-roster match rate; investigate any
unmatched games before trusting downstream numbers.


In [4]:
def iter_annotation_games(annot_root):
    for split_file in sorted(annot_root.rglob("split/*.json")):
        source = split_file.parent.parent.name
        data = json.loads(split_file.read_text(encoding="utf-8"))
        for game in data:
            session = (game.get("video_name") or game.get("session")
                       or game.get("YT_ID") or game.get("EG_ID"))
            game_id = game.get("Game_ID") or game.get("game")
            if session is None or game_id is None:
                continue
            yield source, session, game_id, game.get("Dialogue", [])


pt_rows, seen_keys = [], set()
for source, session, game_id, dialogue in iter_annotation_games(ANNOT_ROOT):
    key = ckey(source, session, game_id)
    if key in seen_keys:            # same game can appear via duplicate files
        continue
    seen_keys.add(key)
    per_speaker = {}
    for utt in dialogue:
        sp = str(utt.get("speaker", "")).strip()
        if not sp:
            continue
        d = per_speaker.setdefault(sp, Counter())
        d["n_utterances"] += 1
        for ann in utt.get("annotation", []):
            if ann in PT_LABELS:
                d[f"pt_{ann.lower().replace(' ', '_')}"] += 1
    for sp, counts in per_speaker.items():
        pt_rows.append({"key": key, "speaker": sp, **counts})
pt_df = pd.DataFrame(pt_rows).fillna(0)
print(f"annotation games loaded: {len(seen_keys)}")

matched = sum(1 for k in seen_keys if k in roster_by_key)
print(f"annotation games matching a roster key: {matched} / {len(seen_keys)}")
missing = [k for k in roster_by_key if k not in seen_keys]
print(f"roster games with NO annotation match: {len(missing)}")
for k in missing[:5]:
    print("   e.g.", k)


annotation games loaded: 199
annotation games matching a roster key: 191 / 199
roster games with NO annotation match: 0


## Family B: directed accusations + identity-claim features


In [5]:
acc_rows = []
for p in sorted(ACC_TARGETS_ROOT.rglob("*.json")):
    try:
        rec = json.loads(p.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        continue
    meta = rec.get("metadata", {})
    if not all(meta.get(k) for k in ("source", "session", "game")):
        continue
    key = ckey(meta["source"], meta["session"], meta["game"])
    for item in rec.get("items", []):
        for rel in item.get("relations", []):
            if rel.get("type") not in ("werewolf", "deception"):
                continue
            for pl in rel.get("accused", []):
                if pl != "UNKNOWN":
                    acc_rows.append({"key": key, "player": pl, "relation_type": rel["type"]})
acc_df = pd.DataFrame(acc_rows)
acc_counts = (acc_df.groupby(["key", "player", "relation_type"]).size().unstack(fill_value=0)
              .reset_index()) if not acc_df.empty else pd.DataFrame(columns=["key", "player"])
for col, new in [("werewolf", "werewolf_count"), ("deception", "deception_count")]:
    if col in acc_counts.columns:
        acc_counts = acc_counts.rename(columns={col: new})
    else:
        acc_counts[new] = 0
print(f"accusation events: {len(acc_df)}; games covered: {acc_counts['key'].nunique() if len(acc_counts) else 0}")

ic = pd.read_csv(IC_FEATURES_CSV)
ic["key"] = [ckey(s, se, g) for s, se, g in zip(ic["source"], ic["session"], ic["game"])]
INFO_ROLES = {"Seer", "Robber", "Troublemaker", "Insomniac"}
def parse_roles(v):
    try:
        return set(json.loads(v)) if isinstance(v, str) else set()
    except json.JSONDecodeError:
        return set()
ic["roles_set"] = ic["roles_claimed"].apply(parse_roles)
ic["claims_werewolf"] = ic["roles_set"].apply(lambda s: int("Werewolf" in s))
ic["made_any_claim"] = ic["roles_set"].apply(lambda s: int(len(s) > 0))
ic["claims_info_role"] = ic["roles_set"].apply(lambda s: int(bool(s & INFO_ROLES)))
ic["n_distinct_roles_claimed_self"] = pd.to_numeric(
    ic["n_distinct_roles_claimed_self"], errors="coerce").fillna(0)
for col in ("is_in_role_conflict", "is_self_contradiction"):
    ic[col] = ic[col].astype(str).str.lower().eq("true").astype(int)
ic_feats = ic[["key", "player", "claims_werewolf", "made_any_claim", "claims_info_role",
               "n_distinct_roles_claimed_self", "is_in_role_conflict", "is_self_contradiction"]]
print(f"identity-claim rows: {len(ic_feats)}; games covered: {ic_feats['key'].nunique()}")


accusation events: 2001; games covered: 182
identity-claim rows: 632; games covered: 189


## Candidate table

One row per (game, roster player), features merged by name (case-insensitive
against the roster). Missing = 0: a player never annotated as doing or
receiving anything genuinely has zero counts.


In [6]:
def name_map(players):
    return {p.strip().lower(): p for p in players}


cand_rows = []
for key, info in roster_by_key.items():
    nmap = name_map(info["players"])

    def canon_player(name):
        return nmap.get(str(name).strip().lower())

    feats = {p: dict.fromkeys(PLAYER_FEATURES, 0.0) for p in info["players"]}
    for df, cols in [(pt_df[pt_df["key"] == key], FAMILY_A),
                     (acc_counts[acc_counts["key"] == key] if len(acc_counts) else pd.DataFrame(),
                      ["werewolf_count", "deception_count"]),
                     (ic_feats[ic_feats["key"] == key],
                      ["claims_werewolf", "made_any_claim", "claims_info_role",
                       "n_distinct_roles_claimed_self", "is_in_role_conflict", "is_self_contradiction"])]:
        for _, r in df.iterrows():
            p = canon_player(r.get("speaker", r.get("player")))
            if p is None:
                continue
            for c in cols:
                if c in r and pd.notna(r[c]):
                    feats[p][c] += float(r[c])

    base = {"key": key, "source": info["source"], "session": info["session"],
            "game": info["game"]}
    zero_circ = dict.fromkeys(CIRCLE_FEATURES, 0.0)
    for p, f in feats.items():
        cand_rows.append({**base, "player": p, **f, **zero_circ})

    # the "No Werewolf" alternative: no player-level activity of its own, but it
    # carries game-level summaries describing how diffuse the suspicion was.
    ww = np.array([f["werewolf_count"] for f in feats.values()], dtype=float)
    circ = {"is_circle_option": 1.0,
            "circ_max_werewolf_count": float(ww.max()) if len(ww) else 0.0,
            "circ_mean_werewolf_count": float(ww.mean()) if len(ww) else 0.0,
            "circ_share_players_accused": float((ww > 0).mean()) if len(ww) else 0.0,
            "circ_any_claims_werewolf": float(max(f["claims_werewolf"] for f in feats.values())),
            "circ_any_role_conflict": float(max(f["is_in_role_conflict"] for f in feats.values()))}
    cand_rows.append({**base, "player": CIRCLE_OPTION,
                      **dict.fromkeys(PLAYER_FEATURES, 0.0), **circ})

candidates_raw = pd.DataFrame(cand_rows)
print(f"candidate rows: {len(candidates_raw)} over {candidates_raw['key'].nunique()} games "
      f"(= players + 1 'No Werewolf' alternative per game)")
print("nonzero feature coverage among player rows (share > 0):")
players_only = candidates_raw[candidates_raw["is_circle_option"] == 0]
display((players_only[PLAYER_FEATURES] > 0).mean().round(3).to_frame("share_nonzero"))


def within_game_scaled(candidates, feature_cols=PLAYER_FEATURES):
    """z-score every player feature WITHIN its game (the circle row stays at 0,
    i.e. 'an average player'). The metric is a within-game argmax, so only a
    player's standing relative to this roster can decide the vote; raw counts
    also carry how chatty the whole game was, which is noise for the task."""
    out = candidates.copy()
    is_player = out["is_circle_option"] == 0
    grp = out.loc[is_player].groupby("key")
    for c in feature_cols:
        mu = grp[c].transform("mean")
        sd = grp[c].transform("std").replace(0.0, np.nan)
        out.loc[is_player, c] = ((out.loc[is_player, c] - mu) / sd).fillna(0.0)
        out.loc[~is_player, c] = 0.0
    return out


CANDIDATES = {"raw": candidates_raw, "within_game": within_game_scaled(candidates_raw)}
print("\nwithin-game scaling sanity check (player rows should be ~mean 0, sd ~1):")
chk = CANDIDATES["within_game"]
chk = chk[chk["is_circle_option"] == 0][["werewolf_count", "pt_defense", "n_utterances"]]
display(chk.agg(["mean", "std"]).round(3))


candidate rows: 1055 over 191 games (= players + 1 'No Werewolf' alternative per game)
nonzero feature coverage among player rows (share > 0):


,share_nonzero
pt_accusation,0.854
pt_defense,0.816
pt_interrogation,0.914
pt_identity_declaration,0.728
pt_evidence,0.832
pt_call_for_action,0.631
n_utterances,0.978
werewolf_count,0.527
deception_count,0.416
claims_werewolf,0.110



within-game scaling sanity check (player rows should be ~mean 0, sd ~1):


,werewolf_count,pt_defense,n_utterances
mean,0.000,-0.000,-0.000
std,0.839,0.876,0.883


## Training rows and evaluation machinery

Each labeled instance expands to its game's candidate rows (label = 1 for the
voted player). CV is grouped by game with shuffled fold assignment per seed.
The metric is instance-level top-1, averaged per game first so every game
weighs equally regardless of how many valid runs it has.


In [7]:
def cols_for(family, include_circle=True):
    """Every family also gets the alternative-specific circle covariates --
    otherwise 'No Werewolf' has no way of ever winning the argmax."""
    return FAMILIES[family] + (CIRCLE_FEATURES if include_circle else [])


def build_training(instances, candidates, model, label_source):
    inst = instances[(instances["model"] == model) & (instances["label_source"] == label_source)]
    by_key = {k: g for k, g in candidates.groupby("key")}
    rows, dropped = [], 0
    for _, r in inst.iterrows():
        cand = by_key.get(r["key"])
        if cand is None or r["voted"] not in set(cand["player"]):
            dropped += 1
            continue
        for _, c in cand.iterrows():
            rows.append({"key": r["key"], "instance": (r["key"], r["run_label"]),
                         "player": c["player"], "label": int(c["player"] == r["voted"]),
                         "true_is_circle": int(r["voted"] == CIRCLE_OPTION),
                         **{f: c[f] for f in ALL_FEATURES}})
    return pd.DataFrame(rows), dropped


def top1_score(df, score_col):
    # per instance: argmax candidate == voted?  then mean per game, then mean over games
    def inst_ok(g):
        return int(g.loc[g[score_col].idxmax(), "label"] == 1)
    per_inst = df.groupby("instance", sort=False).apply(inst_ok, include_groups=False)
    inst_game = pd.DataFrame({"ok": per_inst,
                              "game": [i[0] for i in per_inst.index]})
    return float(inst_game.groupby("game")["ok"].mean().mean())


def top1_breakdown(df, score_col):
    """Overall top-1 plus the split by what the LLM actually chose, so a model
    that simply never predicts 'No Werewolf' cannot hide behind the average."""
    picks = df.loc[df.groupby("instance", sort=False)[score_col].idxmax()]
    out = {"top1": top1_score(df, score_col),
           "pred_circle_rate": float(picks["is_circle_option"].mean())}
    for name, mask in [("top1_player_votes", picks["true_is_circle"] == 0),
                       ("top1_circle_votes", picks["true_is_circle"] == 1)]:
        sub = picks[mask]
        out[name] = float(sub["label"].mean()) if len(sub) else np.nan
        out[name + "_n"] = int(len(sub))
    return out


def shuffled_folds(keys, n_folds, seed):
    keys = list(keys)
    np.random.default_rng(seed).shuffle(keys)
    return {k: i % n_folds for i, k in enumerate(keys)}


def l1_selector(random_state=RANDOM_STATE):
    return LogisticRegressionCV(Cs=5, cv=3, penalty="l1", solver="liblinear",
                                class_weight="balanced", max_iter=2000,
                                random_state=random_state, scoring="neg_log_loss")


class L1SelectedLogReg:
    """Embedded feature selection: an L1 path picks the features on the training
    fold, then a plain (L2) balanced logreg is refit on the survivors only.

    Selection never sees the held-out games, so the CV score is an honest
    estimate of "let the model choose its own feature subset". `support_` is the
    boolean mask of kept features, in feature_cols order.
    """

    def __init__(self, random_state=RANDOM_STATE):
        self.random_state = random_state

    def fit(self, X, y):
        sel = l1_selector(self.random_state).fit(X, y)
        self.support_ = np.abs(sel.coef_[0]) > 1e-8
        if not self.support_.any():          # degenerate path: keep everything
            self.support_ = np.ones(X.shape[1], dtype=bool)
        self.model_ = LogisticRegression(max_iter=1000, class_weight="balanced")
        self.model_.fit(X[:, self.support_], y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X[:, self.support_])


def fit_method(method, X_train, y_train):
    if method == "logreg":
        m = LogisticRegression(max_iter=1000, class_weight="balanced")
    elif method == "gbm":
        # GBM takes no class_weight; the sample weights keep it on the same
        # footing as the balanced linear models.
        m = GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=50,
                                       max_depth=2, subsample=0.8)
        return m.fit(X_train, y_train,
                     sample_weight=compute_sample_weight("balanced", y_train))
    elif method == "l1_select":
        m = L1SelectedLogReg()
    else:
        m = l1_selector()
    m.fit(X_train, y_train)
    return m


def fit_and_score(df, cols, method, tr_mask, te_mask):
    """Fit on tr_mask, score te_mask. The scaler is fit on the training rows
    only. Returns (top-1, scored test frame)."""
    tr, te = df[tr_mask], df[te_mask].copy()
    scaler = StandardScaler().fit(tr[cols])
    model = fit_method(method, scaler.transform(tr[cols]), tr["label"])
    te["score"] = model.predict_proba(scaler.transform(te[cols]))[:, 1]
    return top1_score(te, "score"), te, model, scaler


def cv_evaluate(train_df, feature_cols, methods=None, collect_perm=False):
    """Flat GroupKFold-by-game CV. Returns {method: (mean, std)} and, if
    requested, per-feature held-out permutation drops as
    {method: {feature: [per-fold drops]}}. Pass `methods` to fit only some
    estimators (permutation is expensive, so the importance section passes the
    single winning method)."""
    methods = methods or METHODS
    results = {m: [] for m in methods}
    perm_deltas = {m: {f: [] for f in feature_cols} for m in methods}
    for seed in CV_SEEDS:
        fold_of = shuffled_folds(train_df["key"].unique(), N_FOLDS, seed)
        folds = train_df["key"].map(fold_of)
        for fold in range(N_FOLDS):
            tr_mask, te_mask = folds != fold, folds == fold
            if not te_mask.any() or train_df.loc[tr_mask, "label"].nunique() < 2:
                continue
            for method in methods:
                base, te, model, scaler = fit_and_score(train_df, feature_cols, method,
                                                        tr_mask, te_mask)
                results[method].append(base)
                if collect_perm:
                    X_te = scaler.transform(te[feature_cols])
                    rng = np.random.default_rng(RANDOM_STATE + fold)
                    for j, f in enumerate(feature_cols):
                        X_perm = X_te.copy()
                        X_perm[:, j] = rng.permutation(X_perm[:, j])
                        te["score_p"] = model.predict_proba(X_perm)[:, 1]
                        perm_deltas[method][f].append(base - top1_score(te, "score_p"))
    summary = {m: (float(np.mean(v)), float(np.std(v))) for m, v in results.items() if v}
    return summary, perm_deltas


CONFIG_SPACE = [(fam, meth, sc) for fam in FAMILIES for meth in METHODS for sc in SCALINGS]


def nested_cv(train_by_scaling, config_space=CONFIG_SPACE, include_circle=True):
    """Outer folds report, inner folds choose. The (family, method, scaling)
    triple is selected inside each outer training set, so the reported score
    never sees the configuration search -- this is the number to quote. The
    flat grid elsewhere in the notebook is exploratory and optimistic by
    construction, because its winner is picked on the same folds it reports.
    Also returns the pooled out-of-fold scores for the metric comparison."""
    keys = sorted(train_by_scaling["raw"]["key"].unique())   # tuples: keep as a list
    fold_cols = {sc: {} for sc in train_by_scaling}
    outer_rows, chosen, oof = [], [], []
    for seed in CV_SEEDS:
        fold_of = shuffled_folds(keys, N_FOLDS, seed)
        for sc, df in train_by_scaling.items():
            fold_cols[sc][seed] = df["key"].map(fold_of)
        for fold in range(N_FOLDS):
            tr_keys = [k for k in keys if fold_of[k] != fold]
            inner_of = shuffled_folds(tr_keys, INNER_FOLDS, seed)
            best_cfg, best_inner = None, -np.inf
            for family, method, scaling in config_space:
                df = train_by_scaling[scaling]
                outer_tr = fold_cols[scaling][seed] != fold
                inner_fold = df["key"].map(inner_of)
                scores = []
                for ifold in range(INNER_FOLDS):
                    itr = outer_tr & (inner_fold != ifold)
                    ite = outer_tr & (inner_fold == ifold)
                    if not ite.any() or df.loc[itr, "label"].nunique() < 2:
                        continue
                    scores.append(fit_and_score(df, cols_for(family, include_circle),
                                                method, itr, ite)[0])
                if scores and np.mean(scores) > best_inner:
                    best_inner, best_cfg = float(np.mean(scores)), (family, method, scaling)
            family, method, scaling = best_cfg
            df = train_by_scaling[scaling]
            tr_mask = fold_cols[scaling][seed] != fold
            te_mask = ~tr_mask
            _, te, _, _ = fit_and_score(df, cols_for(family, include_circle), method,
                                        tr_mask, te_mask)
            outer_rows.append({"seed": seed, "fold": fold, "family": family,
                               "method": method, "scaling": scaling,
                               "inner_top1": round(best_inner, 4),
                               **top1_breakdown(te, "score")})
            chosen.append(best_cfg)
            oof.append(te.assign(seed=seed, fold=fold))
    return pd.DataFrame(outer_rows), Counter(chosen), pd.concat(oof, ignore_index=True)


## Baselines

Uniform random (analytic 1/n per game), most-talkative, most-accused
(werewolf-type), and crowd-modal (human village modal target; only defined on
games with >= 2 village voters -- its n is smaller and reported).


In [8]:
village = pd.read_csv(HUMAN_TABLES_DIR / "village_vote_dispersion.csv")
village["key"] = [ckey(*gid.split(" / ")) for gid in village["game_id"]]
crowd_modal = {r["key"]: set(json.loads(r["village_top_target_names"]))
               for _, r in village.iterrows() if r["n_village_aligned_votes"] >= 2}

JITTER = 1e-6      # breaks ties between equally-scored candidates
TIE_DRAWS = 25     # ... averaged over this many independent draws


def expected_top1(df, raw_score, n_draws=TIE_DRAWS, seed=0):
    """Expected top-1 of a rule under uniform random tie-breaking.

    This matters more than it looks: 47% of players have werewolf_count == 0, so
    in a large share of games the most-accused rule is decided entirely by which
    tied candidate the jitter happens to favour. A single draw moved this
    baseline by up to 0.04 between runs -- enough to flip whether the fitted
    model looks better than it. Averaging over draws makes the baseline the
    expected accuracy of the rule instead of one lottery outcome."""
    rng = np.random.default_rng(seed)
    raw_score = np.asarray(raw_score, dtype=float)
    return float(np.mean([top1_score(df.assign(score=raw_score
                                               + rng.uniform(0, JITTER, len(df))), "score")
                          for _ in range(n_draws)]))


def baseline_scores(train_df):
    """Baselines are computed on the RAW-count training frame (they are rules
    about counts, not fitted models). most_talkative / most_accused can never
    pick 'No Werewolf', which is why the circle-aware rule is included: it is
    the simplest rule that can."""
    out = {}
    per_game = train_df.groupby("key")
    out["random_uniform"] = float(np.mean([1.0 / g["player"].nunique() for _, g in per_game]))
    for name, col in [("most_talkative", "n_utterances"), ("most_accused_ww", "werewolf_count")]:
        out[name] = expected_top1(train_df, train_df[col])

    if (train_df["is_circle_option"] == 1).any():
        # "accuse the most-accused player, unless nobody was accused at all"
        circle_rule = np.where(train_df["is_circle_option"] == 1,
                               (train_df["circ_max_werewolf_count"] == 0).astype(float) * 1e3,
                               train_df["werewolf_count"])
        out["most_accused_else_circle"] = expected_top1(train_df, circle_rule)

    cm = train_df[train_df["key"].isin(crowd_modal)]
    if not cm.empty:
        hit = np.array([1.0 if r["player"] in crowd_modal[r["key"]] else 0.0
                        for _, r in cm.iterrows()])
        out["crowd_modal"] = expected_top1(cm, hit)
        out["crowd_modal_n_games"] = cm["key"].nunique()
    return out

## Run everything

In [9]:
cv_rows, base_rows = [], []
train_cache = {}          # (model, label_source, scaling) -> training frame
for model in MODELS:
    for label_source in ["stochastic", "greedy"]:
        for scaling in SCALINGS:
            df, dropped = build_training(instances, CANDIDATES[scaling], model, label_source)
            if df.empty:
                continue
            train_cache[(model, label_source, scaling)] = df
            if scaling == "raw":
                b = baseline_scores(df)
                base_rows.append({"model": model, "label_source": label_source,
                                  "n_games": df["key"].nunique(),
                                  "n_instances": df["instance"].nunique(),
                                  "share_circle_instances": round(
                                      df.groupby("instance")["true_is_circle"].first().mean(), 3),
                                  "dropped_instances": dropped, **b})
            for family in FAMILIES:
                summary, _ = cv_evaluate(df, cols_for(family))
                for method, (mean, std) in summary.items():
                    cv_rows.append({"model": model, "label_source": label_source,
                                    "family": family, "method": method, "scaling": scaling,
                                    "top1_mean": round(mean, 3), "top1_std": round(std, 3)})

cv_results = pd.DataFrame(cv_rows)
baselines = pd.DataFrame(base_rows)
print("Baselines (choice set = roster + 'No Werewolf'):")
display(baselines.round(3))
print("\nFlat CV grid (top-1, mean over 5 folds x 3 seeds) -- EXPLORATORY, see nested CV below:")
display(cv_results[cv_results["label_source"] == IMPORTANCE_LABEL_SOURCE]
        .pivot_table(index=["model", "family"], columns=["scaling", "method"],
                     values="top1_mean").round(3))


Baselines (choice set = roster + 'No Werewolf'):


,model,label_source,n_games,n_instances,share_circle_instances,dropped_instances,random_uniform,most_talkative,most_accused_ww,most_accused_else_circle,crowd_modal,crowd_modal_n_games
0,2B,stochastic,191,568,0.231,0,0.184,0.146,0.365,0.399,0.383,169
1,2B,greedy,191,191,0.251,0,0.184,0.129,0.354,0.395,0.362,169
2,31B,stochastic,191,573,0.120,0,0.184,0.187,0.401,0.408,0.456,169
3,31B,greedy,191,191,0.126,0,0.184,0.201,0.380,0.387,0.441,169
4,4B,stochastic,191,573,0.059,0,0.184,0.214,0.403,0.403,0.429,169
5,4B,greedy,191,191,0.094,0,0.184,0.184,0.430,0.438,0.429,169



Flat CV grid (top-1, mean over 5 folds x 3 seeds) -- EXPLORATORY, see nested CV below:


scaling              raw                         within_game                   \
method               gbm l1_select  lasso logreg         gbm l1_select  lasso   
model family                                                                    
2B    A_pt         0.293     0.314  0.313  0.314       0.248     0.294  0.296   
      B_directed   0.470     0.480  0.477  0.480       0.445     0.469  0.467   
      C_combined   0.467     0.470  0.471  0.470       0.443     0.478  0.478   
      D_deductive  0.387     0.393  0.395  0.393       0.383     0.408  0.407   
      S_social     0.400     0.411  0.411  0.411       0.371     0.388  0.390   
31B   A_pt         0.216     0.237  0.234  0.237       0.205     0.226  0.228   
      B_directed   0.393     0.419  0.419  0.419       0.404     0.412  0.412   
      C_combined   0.396     0.416  0.417  0.416       0.406     0.410  0.412   
      D_deductive  0.259     0.291  0.291  0.291       0.260     0.298  0.296   
      S_social     0.368     0.379  0.379  0.379       0.372     0.408  0.407   
4B    A_pt         0.243     0.287  0.283  0.287       0.243     0.271  0.272   
      B_directed   0.467     0.460  0.463  0.460       0.510     0.496  0.496   
      C_combined   0.462     0.454  0.454  0.454       0.488     0.489  0.492   
      D_deductive  0.373     0.356  0.352  0.355       0.372     0.359  0.361   
      S_social     0.345     0.372  0.374  0.372       0.368     0.392  0.389   

scaling                   
method            logreg  
model family              
2B    A_pt         0.294  
      B_directed   0.469  
      C_combined   0.478  
      D_deductive  0.408  
      S_social     0.388  
31B   A_pt         0.226  
      B_directed   0.412  
      C_combined   0.410  
      D_deductive  0.298  
      S_social     0.408  
4B    A_pt         0.271  
      B_directed   0.495  
      C_combined   0.489  
      D_deductive  0.359  
      S_social     0.392

## Nested CV -- the number to quote

The grid above has a problem that no caveat fixes: its winner is chosen by
looking at the same folds that then report its score. With 40 configurations
(5 families x 4 methods x 2 scalings) the maximum of a noisy grid is biased
upwards, and the winner-vs-runner-up gaps here are far inside one fold std.

Nested CV separates the two jobs. Inside each outer training set, an inner
3-fold CV picks the configuration; the outer fold then scores that choice and
is never consulted again. The result is one honest number per LLM plus a
record of *which* configuration the selection keeps landing on -- if it is
unstable across folds, that instability is itself the finding.

Reported alongside: `top1_player_votes` / `top1_circle_votes` (accuracy split
by what the LLM actually chose) and `pred_circle_rate`, because a surrogate
that never predicts "No Werewolf" would otherwise look fine on the average.


In [10]:
nested_rows, nested_choice_rows, oof_by_model = [], [], {}
for model in MODELS:
    by_scaling = {sc: train_cache[(model, IMPORTANCE_LABEL_SOURCE, sc)] for sc in SCALINGS}
    folds_df, chosen, oof = nested_cv(by_scaling)
    oof_by_model[model] = oof
    agg = {c: folds_df[c].mean() for c in ["top1", "top1_player_votes",
                                           "top1_circle_votes", "pred_circle_rate"]}
    nested_rows.append({"model": model, "label_source": IMPORTANCE_LABEL_SOURCE,
                        "nested_top1_mean": round(agg["top1"], 3),
                        "nested_top1_std": round(folds_df["top1"].std(), 3),
                        "top1_player_votes": round(agg["top1_player_votes"], 3),
                        "top1_circle_votes": round(agg["top1_circle_votes"], 3),
                        "pred_circle_rate": round(agg["pred_circle_rate"], 3),
                        "flat_grid_best": cv_results[
                            (cv_results["model"] == model)
                            & (cv_results["label_source"] == IMPORTANCE_LABEL_SOURCE)
                        ]["top1_mean"].max()})
    for (fam, meth, sc), n in chosen.most_common():
        nested_choice_rows.append({"model": model, "family": fam, "method": meth,
                                   "scaling": sc, "n_outer_folds": n,
                                   "share": round(n / len(folds_df), 2)})

nested_results = pd.DataFrame(nested_rows)
nested_choices = pd.DataFrame(nested_choice_rows)
print("Nested CV -- honest top-1 per LLM (and the optimistic flat-grid maximum for contrast):")
display(nested_results)
print("\nWhich configuration the inner CV selected, across the 15 outer folds:")
display(nested_choices)


Nested CV -- honest top-1 per LLM (and the optimistic flat-grid maximum for contrast):


,model,label_source,nested_top1_mean,nested_top1_std,top1_player_votes,top1_circle_votes,pred_circle_rate,flat_grid_best
0,2B,stochastic,0.471,0.061,0.462,0.508,0.280,0.480
1,31B,stochastic,0.405,0.080,0.418,0.363,0.129,0.419
2,4B,stochastic,0.492,0.065,0.518,0.073,0.021,0.510



Which configuration the inner CV selected, across the 15 outer folds:


,model,family,method,scaling,n_outer_folds,share
0,2B,C_combined,logreg,raw,7,0.47
1,2B,B_directed,gbm,within_game,2,0.13
2,2B,C_combined,logreg,within_game,2,0.13
3,2B,C_combined,lasso,within_game,1,0.07
4,2B,B_directed,logreg,raw,1,0.07
5,2B,B_directed,lasso,within_game,1,0.07
6,2B,C_combined,gbm,raw,1,0.07
7,31B,C_combined,logreg,raw,4,0.27
8,31B,B_directed,logreg,within_game,4,0.27
9,31B,B_directed,lasso,within_game,2,0.13


## Held-out permutation importance -- best surrogate per LLM only

Importance is only meaningful for a surrogate that actually predicts the vote,
so it is computed for **one configuration per LLM**: the (family, method,
scaling) that the nested inner CV selected in the most outer folds, on the
**stochastic** label source only (3x the instances of greedy, so the per-fold
drops are much less noisy).

Each of the 15 train/test splits is refit, and every feature is shuffled
*within the held-out games only*; the reported importance is the mean top-1
drop with its across-fold std, plus the share of folds where the drop was
positive (a stability check -- ~0.5 means the feature is noise).


In [11]:
best_rows, perm_rows = [], []
for model in MODELS:
    ch = nested_choices[nested_choices["model"] == model]
    if ch.empty:
        continue
    # most-selected config; ties broken deterministically so a rerun is stable
    ch = ch.sort_values(["n_outer_folds", "family", "method", "scaling"],
                        ascending=[False, True, True, True])
    modal = ch.iloc[0]
    if modal["share"] < 0.34:
        print(f"NOTE: {model}'s nested selection is unstable -- the most-selected "
              f"configuration only won {modal['n_outer_folds']} of the outer folds. "
              "Read the importances below as 'a good model's view', not 'the' model's.")
    family, method, scaling = modal["family"], modal["method"], modal["scaling"]
    train_df = train_cache[(model, IMPORTANCE_LABEL_SOURCE, scaling)]
    cols = cols_for(family)
    _, perm = cv_evaluate(train_df, cols, methods=[method], collect_perm=True)
    for f, deltas in perm[method].items():
        d = np.asarray(deltas, dtype=float)
        perm_rows.append({"model": model, "label_source": IMPORTANCE_LABEL_SOURCE,
                          "family": family, "method": method, "scaling": scaling,
                          "feature": f, "top1_drop_mean": round(float(d.mean()), 4),
                          "top1_drop_std": round(float(d.std()), 4),
                          "share_folds_positive": round(float((d > 0).mean()), 2)})
    nested_top1 = nested_results.loc[nested_results["model"] == model,
                                     "nested_top1_mean"].iloc[0]
    best_rows.append({"model": model, "label_source": IMPORTANCE_LABEL_SOURCE,
                      "family": family, "method": method, "scaling": scaling,
                      "selected_in_n_of_15_folds": int(modal["n_outer_folds"]),
                      "nested_top1": nested_top1})

best_df = pd.DataFrame(best_rows)
perm_df = pd.DataFrame(perm_rows)
print("Configuration interpreted per LLM = the one the nested inner CV picked most often:")
display(best_df)

for _, b in best_df.iterrows():
    sub = (perm_df[perm_df["model"] == b["model"]]
           .sort_values("top1_drop_mean", ascending=False)
           [["feature", "top1_drop_mean", "top1_drop_std", "share_folds_positive"]])
    print(f"\n{b['model']} -- {b['family']} / {b['method']} / {b['scaling']} "
          f"(nested top-1 {b['nested_top1']:.3f}); mean top-1 drop when the feature is shuffled:")
    display(sub.reset_index(drop=True))


NOTE: 31B's nested selection is unstable -- the most-selected configuration only won 4 of the outer folds. Read the importances below as 'a good model's view', not 'the' model's.


Configuration interpreted per LLM = the one the nested inner CV picked most often:


,model,label_source,family,method,scaling,selected_in_n_of_15_folds,nested_top1
0,2B,stochastic,C_combined,logreg,raw,7,0.471
1,31B,stochastic,B_directed,logreg,within_game,4,0.405
2,4B,stochastic,B_directed,gbm,within_game,6,0.492



2B -- C_combined / logreg / raw (nested top-1 0.471); mean top-1 drop when the feature is shuffled:


,feature,top1_drop_mean,top1_drop_std,share_folds_positive
0,werewolf_count,0.1223,0.0447,1.00
1,claims_werewolf,0.1042,0.0424,1.00
2,is_circle_option,0.1026,0.0599,1.00
3,circ_max_werewolf_count,0.0758,0.0390,1.00
4,circ_share_players_accused,0.0547,0.0460,0.87
5,n_utterances,0.0145,0.0343,0.60
6,circ_mean_werewolf_count,0.0142,0.0309,0.73
7,pt_defense,0.0100,0.0304,0.53
8,circ_any_claims_werewolf,0.0087,0.0172,0.47
9,claims_info_role,0.0018,0.0176,0.47



31B -- B_directed / logreg / within_game (nested top-1 0.405); mean top-1 drop when the feature is shuffled:


,feature,top1_drop_mean,top1_drop_std,share_folds_positive
0,werewolf_count,0.1394,0.0675,1.00
1,circ_mean_werewolf_count,0.0525,0.0614,0.73
2,n_distinct_roles_claimed_self,0.0450,0.0621,0.73
3,claims_werewolf,0.0209,0.0321,0.73
4,made_any_claim,0.0042,0.0420,0.47
5,is_self_contradiction,0.0036,0.0289,0.33
6,circ_share_players_accused,0.0028,0.0161,0.40
7,circ_max_werewolf_count,0.0018,0.0209,0.40
8,circ_any_claims_werewolf,0.0001,0.0142,0.40
9,is_in_role_conflict,-0.0000,0.0200,0.33



4B -- B_directed / gbm / within_game (nested top-1 0.492); mean top-1 drop when the feature is shuffled:


,feature,top1_drop_mean,top1_drop_std,share_folds_positive
0,claims_werewolf,0.1675,0.0643,1.00
1,werewolf_count,0.1221,0.0596,1.00
2,deception_count,0.0245,0.0337,0.67
3,circ_share_players_accused,0.0082,0.0126,0.40
4,n_distinct_roles_claimed_self,0.0064,0.0207,0.53
5,made_any_claim,0.0064,0.0178,0.40
6,claims_info_role,0.0058,0.0180,0.53
7,is_in_role_conflict,0.0047,0.0131,0.47
8,circ_mean_werewolf_count,0.0029,0.0142,0.40
9,circ_max_werewolf_count,0.0024,0.0093,0.27


## Which features does the model pick on its own?

The hand-drawn families (A / B / C / S / D) impose *my* grouping. Three of the
four estimators do not need it -- they can select features themselves:

- **`lasso`** -- L1 logistic regression, penalty tuned by inner CV; irrelevant
  features get exactly zero coefficients.
- **`l1_select`** -- the same L1 path used purely as a *selector* inside each
  training fold, then an unpenalised logreg refit on the survivors. Selection
  never touches the held-out games.
- **`gbm`** -- not sparse by construction, but depth-2 stumps simply never
  split on useless features, so split-gain shares act as a soft selection.
- **`logreg`** (L2) is the only one forced to use everything.

**And here they have nothing to select away.** With 15 features and ~2400
candidate rows the inner CV lands on a weak penalty, L1 keeps all 15 features
in all 15 folds, and `lasso` / `l1_select` / `logreg` return *identical* top-1
in every cell of the results table above. That is itself the answer to "can a
model do the selection for me": yes, but at this n/p ratio automatic selection
is inert -- the feature set is already small enough that nothing is worth
dropping, and any subset story has to come from the ranking, not from zeros.

So the ranking below comes from the **L1 regularisation path**: fit at 25
penalties from very harsh to very weak and record when each feature first
becomes nonzero. Entering early = surviving the harshest penalty = the model's
own relevance ordering, and unlike a single tuned fit it does not depend on
where inner CV happens to put C. Reported per fold and averaged over the 15
training folds, next to the mean L1 coefficient (sign = direction of the
effect on being voted) and the GBM split-gain share.

This is where a cross-family subset such as *directed accusations +
`pt_defense`* shows up if it exists -- without hypothesising it in advance.


In [12]:
C_GRID = np.logspace(-3, 1, 25)   # strong -> weak L1 penalty
# one scaling for all three LLMs so the entry ranks are comparable across them
SELECTION_SCALING = best_df["scaling"].mode().iloc[0] if not best_df.empty else "within_game"
SELECTION_COLS = cols_for("C_combined")


def l1_entry_order(X, y, feature_cols):
    """Walk the L1 path from a penalty that zeroes everything to one that keeps
    everything, and record the C at which each feature first becomes nonzero.
    Entering early = the feature survives the harshest penalty = the model's own
    ranking of relevance, and unlike a single fit it does not depend on where
    inner CV happens to put C."""
    entry = {f: np.inf for f in feature_cols}
    for C in C_GRID:
        coef = LogisticRegression(penalty="l1", solver="liblinear", C=C,
                                  class_weight="balanced", max_iter=2000).fit(X, y).coef_[0]
        for j, f in enumerate(feature_cols):
            if entry[f] == np.inf and abs(coef[j]) > 1e-8:
                entry[f] = C
    return entry


def selection_report(train_df, feature_cols):
    """Per training fold: the L1 path entry order, whether the CV-tuned L1 fit
    zeroes anything at all, and the GBM split-gain share. All fit on training
    folds only -- a stability report over 15 independent fits."""
    per_fold, n_kept, chosen_C = [], [], []
    for seed in CV_SEEDS:
        fold_of = shuffled_folds(train_df["key"].unique(), N_FOLDS, seed)
        folds = train_df["key"].map(fold_of)
        for fold in range(N_FOLDS):
            tr = train_df[folds != fold]
            if tr["label"].nunique() < 2:
                continue
            X = StandardScaler().fit_transform(tr[feature_cols])
            y = tr["label"]
            sel = l1_selector().fit(X, y)
            coef = sel.coef_[0]
            gain = fit_method("gbm", X, y).feature_importances_
            rank = pd.Series(l1_entry_order(X, y, feature_cols)).rank(method="min")
            n_kept.append(int((np.abs(coef) > 1e-8).sum()))
            chosen_C.append(float(sel.C_[0]))
            for j, f in enumerate(feature_cols):
                per_fold.append({"feature": f, "coef": coef[j],
                                 "selected": int(abs(coef[j]) > 1e-8),
                                 "gbm_gain": gain[j], "entry_rank": rank[f]})
    df = pd.DataFrame(per_fold)
    out = (df.groupby("feature")
             .agg(mean_entry_rank=("entry_rank", "mean"),
                  share_enters_first5=("entry_rank", lambda s: float((s <= 5).mean())),
                  mean_l1_coef=("coef", "mean"),
                  gbm_gain_share=("gbm_gain", "mean"),
                  share_folds_selected=("selected", "mean"))
             .reset_index())
    meta = {"mean_n_kept_at_cv_C": float(np.mean(n_kept)),
            "median_cv_C": float(np.median(chosen_C))}
    return out.sort_values("mean_entry_rank"), meta


sel_rows = []
for model in MODELS:
    train_df = train_cache.get((model, IMPORTANCE_LABEL_SOURCE, SELECTION_SCALING))
    if train_df is None:
        continue
    rep, meta = selection_report(train_df, SELECTION_COLS)
    rep.insert(0, "model", model)
    sel_rows.append(rep)
    print(f"\n{model} -- L1 on C_combined + circle covariates ({len(SELECTION_COLS)} features, "
          f"{SELECTION_SCALING} scaling). At the CV-tuned penalty (median C="
          f"{meta['median_cv_C']:.3g}) it keeps {meta['mean_n_kept_at_cv_C']:.1f} features on "
          "average -- so the ranking below comes from the path, not from that single fit.")
    display(rep.drop(columns="model").round(3).reset_index(drop=True))
selection_df = pd.concat(sel_rows, ignore_index=True) if sel_rows else pd.DataFrame()

if not selection_df.empty:
    print("\nL1 path entry rank across LLMs (1 = survives the harshest penalty; "
          "mean over 15 folds):")
    display(selection_df.pivot_table(index="feature", columns="model",
                                     values="mean_entry_rank")
            .loc[SELECTION_COLS].round(1))



2B -- L1 on C_combined + circle covariates (21 features, within_game scaling). At the CV-tuned penalty (median C=1e+04) it keeps 21.0 features on average -- so the ranking below comes from the path, not from that single fit.


,feature,mean_entry_rank,share_enters_first5,mean_l1_coef,gbm_gain_share,share_folds_selected
0,werewolf_count,1.200,1.000,0.763,0.412,1.0
1,claims_werewolf,1.467,1.000,0.642,0.243,1.0
2,is_circle_option,3.000,1.000,0.923,0.024,1.0
3,circ_any_claims_werewolf,4.933,0.733,-0.205,0.009,1.0
4,deception_count,5.000,0.800,0.183,0.038,1.0
5,claims_info_role,5.467,0.467,-0.156,0.018,1.0
6,circ_max_werewolf_count,5.867,0.400,-0.685,0.018,1.0
7,circ_share_players_accused,9.200,0.000,-0.575,0.010,1.0
8,is_self_contradiction,9.333,0.000,-0.158,0.003,1.0
9,pt_interrogation,10.667,0.067,-0.004,0.030,1.0



31B -- L1 on C_combined + circle covariates (21 features, within_game scaling). At the CV-tuned penalty (median C=1e+04) it keeps 21.0 features on average -- so the ranking below comes from the path, not from that single fit.


,feature,mean_entry_rank,share_enters_first5,mean_l1_coef,gbm_gain_share,share_folds_selected
0,werewolf_count,1.000,1.000,0.684,0.443,1.0
1,claims_werewolf,2.000,1.000,0.350,0.128,1.0
2,deception_count,3.067,1.000,0.237,0.068,1.0
3,circ_mean_werewolf_count,3.867,0.933,-0.676,0.030,1.0
4,circ_any_claims_werewolf,4.533,0.867,-0.232,0.011,1.0
5,claims_info_role,6.400,0.067,-0.237,0.025,1.0
6,n_distinct_roles_claimed_self,9.133,0.133,0.575,0.016,1.0
7,pt_accusation,9.267,0.067,0.139,0.017,1.0
8,pt_interrogation,9.667,0.000,-0.051,0.045,1.0
9,pt_defense,10.800,0.067,0.056,0.043,1.0



4B -- L1 on C_combined + circle covariates (21 features, within_game scaling). At the CV-tuned penalty (median C=1e+04) it keeps 20.7 features on average -- so the ranking below comes from the path, not from that single fit.


,feature,mean_entry_rank,share_enters_first5,mean_l1_coef,gbm_gain_share,share_folds_selected
0,claims_werewolf,1.000,1.000,0.809,0.352,1.000
1,werewolf_count,1.600,1.000,0.645,0.347,1.000
2,circ_share_players_accused,3.133,1.000,-0.271,0.012,1.000
3,circ_any_claims_werewolf,4.000,0.933,-0.675,0.018,1.000
4,deception_count,4.667,0.800,0.191,0.052,1.000
5,circ_max_werewolf_count,5.333,0.600,-0.839,0.005,1.000
6,is_in_role_conflict,8.067,0.000,-0.155,0.016,1.000
7,made_any_claim,8.333,0.067,-0.293,0.005,1.000
8,n_utterances,9.133,0.000,0.141,0.020,1.000
9,claims_info_role,10.867,0.067,-0.066,0.014,1.000



L1 path entry rank across LLMs (1 = survives the harshest penalty; mean over 15 folds):


model,2B,31B,4B
feature,,,
pt_accusation,11.2,9.3,15.2
pt_defense,11.6,10.8,13.9
pt_interrogation,10.7,9.7,14.5
pt_identity_declaration,15.9,16.9,15.1
pt_evidence,15.3,11.4,14.7
pt_call_for_action,11.3,14.1,14.1
n_utterances,10.7,15.7,9.1
werewolf_count,1.2,1.0,1.6
deception_count,5.0,3.1,4.7


## Choice framing vs binary framing

An obvious alternative is Lai et al.'s framing: a binary label per pair, "does
A vote B?". It is worth being precise about what that would and would not
change here.

**It is not a different model.** The estimator already *is* a binary
classifier over (instance, candidate) rows -- `label` is 1 for the candidate
the LLM chose and 0 for the rest, and every method above is fit on exactly
that. Only the *evaluation* differs: the choice framing takes one argmax per
instance, the binary framing scores every pair independently. So the two are
two readings of the same fitted scores, and the cell below reports both from
the identical out-of-fold predictions.

**It is not what brings the circle games back either.** They come back because
"No Werewolf" is now an *alternative in the choice set* -- which is what it
actually is on the ballot: the LLM picks one option out of {roster} + {No
Werewolf}. Under the binary framing a circle game would enter as a row of all
zeros -- "the LLM voted for nobody" -- which throws away the information that
it made a positive choice, and leaves the model unable to ever predict that
choice. The choice-set version keeps it as a prediction the model can get
right or wrong, which is why `top1_circle_votes` is reportable at all.

**Why the choice framing is the primary one.** There is exactly one voter per
game per LLM, so "does A vote B" collapses to "does the LLM vote B" -- the
pairwise structure of Lai et al. (many human voters per game) does not exist
in our setup. And the metric has to match the decision: the LLM emits one vote,
so one decision per instance is the unit, not `n_candidates` independent
coin-flips whose base rate (~1 positive in 5-6 rows) makes F1 depend mostly on
the threshold.

**Why the binary numbers are still reported.** Lai et al. report F1 0.33 on
their pairwise task, and a reader will want the comparison. The table below
gives F1 under the argmax rule and PR-AUC (threshold-free) so the number
exists -- but it is *not* the same task: their voters are humans, ours is an
LLM, and their choice sets exclude the circle option. Quote it as context, not
as a head-to-head.


In [13]:
# Same out-of-fold predictions, read two ways.
binary_rows = []
for model, oof in oof_by_model.items():
    o = oof.copy()
    # each instance appears once per seed; keep the game key first in the tuple
    # so the per-game averaging inside top1_score still works
    o["instance"] = [(i[0], i[1], s) for i, s in zip(o["instance"], o["seed"])]
    # choice framing: one decision per instance, argmax over the choice set
    choice = top1_breakdown(o, "score")
    # binary framing: one decision per (instance, candidate) pair -- "does the
    # LLM vote for this candidate?". Same scores, different unit of analysis.
    picks = o.groupby("instance", sort=False)["score"].idxmax()
    y_pred = np.zeros(len(o), dtype=int)
    y_pred[o.index.get_indexer(picks)] = 1
    binary_rows.append({
        "model": model,
        "top1_choice_framing": round(choice["top1"], 3),
        "binary_f1_argmax_rule": round(f1_score(o["label"], y_pred), 3),
        "binary_pr_auc": round(average_precision_score(o["label"], o["score"]), 3),
        "positive_rate": round(o["label"].mean(), 3),
        "n_pairs": len(o), "n_instances": o["instance"].nunique()})
binary_framing = pd.DataFrame(binary_rows)
print("The same nested-CV out-of-fold scores, scored under both framings:")
display(binary_framing)


The same nested-CV out-of-fold scores, scored under both framings:


,model,top1_choice_framing,binary_f1_argmax_rule,binary_pr_auc,positive_rate,n_pairs,n_instances
0,2B,0.471,0.472,0.436,0.181,9402,1704
1,31B,0.405,0.405,0.397,0.181,9495,1719
2,4B,0.492,0.492,0.479,0.181,9495,1719


## Decomposing the vote: abstain, then choose

The full-ballot result above is the complete decision, and it is the headline.
But that decision has two parts, and they are worth separating because they
fail differently:

1. **Abstain or accuse** -- does the LLM name a player at all, or answer "No
   Werewolf"? Reported below as accuracy / precision / recall on the circle
   class, from the same nested-CV out-of-fold predictions.
2. **Given an accusation, whom?** -- the choice among players only. This is the
   *conditional* task: circle instances dropped, circle alternative removed
   from every choice set, circle covariates removed from every model. It is
   re-run through the identical nested CV so the two numbers sit on the same
   methodological footing.

Part 2 is also, exactly, the task the earlier version of this notebook was
measuring. Reporting it as a conditional result is legitimate; reporting it as
*the* result is not, and the reason is in the conditioning set. Each LLM is
conditioned on a different subset of games -- 2B on 77% of its votes, 31B on
88%, 4B on 94% -- and those subsets are not random: an LLM abstains precisely
when suspicion is diffuse, which are plausibly the harder games. So the
conditional number is inflated relative to the full task by an amount that
differs per model, and **conditional scores should not be compared across
LLMs**. Within a model, part 1 x part 2 is the decomposition; across models,
only the full-ballot number is comparable.

In [14]:
# Part 1: the abstain-or-accuse decision, read off the full-ballot OOF scores.
abstain_rows = []
for model, oof in oof_by_model.items():
    o = oof.copy()
    o["instance"] = [(i[0], i[1], s) for i, s in zip(o["instance"], o["seed"])]
    picks = o.loc[o.groupby("instance", sort=False)["score"].idxmax()]
    y_true = picks["true_is_circle"].astype(int)
    y_pred = picks["is_circle_option"].astype(int)
    abstain_rows.append({
        "model": model,
        "accuracy": round(float((y_true == y_pred).mean()), 3),
        "precision_circle": round(float(precision_score(y_true, y_pred, zero_division=0)), 3),
        "recall_circle": round(float(recall_score(y_true, y_pred, zero_division=0)), 3),
        "true_circle_rate": round(float(y_true.mean()), 3),
        "pred_circle_rate": round(float(y_pred.mean()), 3),
        "n_circle_instances": int(y_true.sum())})
abstention = pd.DataFrame(abstain_rows)
print("Part 1 -- predicting whether the LLM abstains ('No Werewolf') at all:")
display(abstention)

Part 1 -- predicting whether the LLM abstains ('No Werewolf') at all:


,model,accuracy,precision_circle,recall_circle,true_circle_rate,pred_circle_rate,n_circle_instances
0,2B,0.722,0.415,0.504,0.231,0.280,393
1,31B,0.825,0.288,0.309,0.120,0.129,207
2,4B,0.929,0.222,0.078,0.059,0.021,102


In [15]:
# Part 2: the conditional task -- players only, same nested CV.
cond_instances = instances[instances["is_circle"] == 0]
COND_CANDIDATES = {sc: c[c["is_circle_option"] == 0].copy() for sc, c in CANDIDATES.items()}

cond_rows, cond_choice_rows, cond_base_rows = [], [], []
for model in MODELS:
    by_scaling = {}
    for sc in SCALINGS:
        df, _ = build_training(cond_instances, COND_CANDIDATES[sc], model,
                               IMPORTANCE_LABEL_SOURCE)
        by_scaling[sc] = df
    cond_base_rows.append({"model": model,
                           "n_instances": by_scaling["raw"]["instance"].nunique(),
                           **baseline_scores(by_scaling["raw"])})
    folds_df, chosen, _ = nested_cv(by_scaling, include_circle=False)
    cond_rows.append({"model": model, "label_source": IMPORTANCE_LABEL_SOURCE,
                      "nested_top1_mean": round(folds_df["top1"].mean(), 3),
                      "nested_top1_std": round(folds_df["top1"].std(), 3),
                      "n_instances": by_scaling["raw"]["instance"].nunique()})
    for (fam, meth, sc), n in chosen.most_common():
        cond_choice_rows.append({"model": model, "family": fam, "method": meth,
                                 "scaling": sc, "n_outer_folds": n,
                                 "share": round(n / len(folds_df), 2)})

cond_nested = pd.DataFrame(cond_rows)
cond_choices = pd.DataFrame(cond_choice_rows)
cond_baselines = pd.DataFrame(cond_base_rows)
print("Part 2 -- conditional on the LLM naming a player, which player (nested CV):")
display(cond_nested)
print("\nBaselines on the conditional task:")
display(cond_baselines.round(3))
print("\nConfigurations the inner CV selected on the conditional task:")
display(cond_choices)

Part 2 -- conditional on the LLM naming a player, which player (nested CV):


,model,label_source,nested_top1_mean,nested_top1_std,n_instances
0,2B,stochastic,0.529,0.077,437
1,31B,stochastic,0.467,0.075,504
2,4B,stochastic,0.526,0.103,539



Baselines on the conditional task:


,model,n_instances,random_uniform,most_talkative,most_accused_ww,crowd_modal,crowd_modal_n_games
0,2B,437,0.226,0.185,0.459,0.470,157
1,31B,504,0.227,0.221,0.450,0.515,158
2,4B,539,0.227,0.224,0.430,0.443,168



Configurations the inner CV selected on the conditional task:


,model,family,method,scaling,n_outer_folds,share
0,2B,C_combined,gbm,within_game,4,0.27
1,2B,B_directed,logreg,within_game,3,0.20
2,2B,C_combined,gbm,raw,2,0.13
3,2B,B_directed,lasso,raw,1,0.07
4,2B,B_directed,gbm,within_game,1,0.07
5,2B,C_combined,logreg,raw,1,0.07
6,2B,C_combined,lasso,raw,1,0.07
7,2B,C_combined,logreg,within_game,1,0.07
8,2B,B_directed,gbm,raw,1,0.07
9,31B,B_directed,logreg,within_game,3,0.20


In [16]:
# The two numbers side by side, with what separates them made explicit.
comparison = (nested_results[["model", "nested_top1_mean", "nested_top1_std"]]
              .rename(columns={"nested_top1_mean": "full_ballot_top1",
                               "nested_top1_std": "full_ballot_std"})
              .merge(cond_nested[["model", "nested_top1_mean", "nested_top1_std",
                                  "n_instances"]]
                     .rename(columns={"nested_top1_mean": "conditional_top1",
                                      "nested_top1_std": "conditional_std",
                                      "n_instances": "conditional_n"}), on="model")
              .merge(baselines[baselines["label_source"] == IMPORTANCE_LABEL_SOURCE]
                     [["model", "n_instances", "share_circle_instances"]]
                     .rename(columns={"n_instances": "full_ballot_n"}), on="model"))
comparison["share_conditioned_away"] = comparison["share_circle_instances"]
comparison = comparison.drop(columns="share_circle_instances")
print("Full ballot (comparable across LLMs) vs conditional (NOT comparable across LLMs,\n"
      "because each is computed on a different subset of that LLM's own decisions):")
display(comparison)

Full ballot (comparable across LLMs) vs conditional (NOT comparable across LLMs,
because each is computed on a different subset of that LLM's own decisions):


,model,full_ballot_top1,full_ballot_std,conditional_top1,conditional_std,conditional_n,full_ballot_n,share_conditioned_away
0,2B,0.471,0.061,0.529,0.077,437,568,0.231
1,31B,0.405,0.080,0.467,0.075,504,573,0.120
2,4B,0.492,0.065,0.526,0.103,539,573,0.059


In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
saved = []
for name, df in [("surrogate_cv_results", cv_results),
                 ("surrogate_baselines", baselines),
                 ("surrogate_nested_cv", nested_results),
                 ("surrogate_nested_cv_selected_configs", nested_choices),
                 ("surrogate_best_config", best_df),
                 ("surrogate_permutation_importance", perm_df),
                 ("surrogate_l1_feature_selection", selection_df),
                 ("surrogate_framing_comparison", binary_framing),
                 ("surrogate_abstention_decision", abstention),
                 ("surrogate_conditional_nested_cv", cond_nested),
                 ("surrogate_conditional_selected_configs", cond_choices),
                 ("surrogate_conditional_baselines", cond_baselines),
                 ("surrogate_task_comparison", comparison)]:
    if df is not None and not df.empty:
        df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
        saved.append(name)
print("saved ->", OUTPUT_DIR.relative_to(REPO_ROOT))
for n in saved:
    print("  ", n + ".csv")


saved -> analysis\cross_model\voting\prompt_v4\predictive\tables
   surrogate_cv_results.csv
   surrogate_baselines.csv
   surrogate_nested_cv.csv
   surrogate_nested_cv_selected_configs.csv
   surrogate_best_config.csv
   surrogate_permutation_importance.csv
   surrogate_l1_feature_selection.csv
   surrogate_framing_comparison.csv
   surrogate_abstention_decision.csv
   surrogate_conditional_nested_cv.csv
   surrogate_conditional_selected_configs.csv
   surrogate_conditional_baselines.csv
   surrogate_task_comparison.csv


## Reading guide and caveats

- **Quote the nested-CV number.** The flat grid is exploratory: its winner is
  chosen on the same folds that report it, so its maximum is biased upwards.
  The gap between the two is printed side by side precisely so the size of that
  bias is visible rather than argued about.
- A model beating **crowd-modal** would mean dialogue features predict the LLM
  vote better than human interpretive consensus does; matching it means LLMs
  and humans read the same surface signals. Neither is causal.
- Family comparisons (A vs B vs C) are the RQ-relevant contrast: whether the
  vote is better explained by *who persuades* (A) or by *what is said about
  whom* (B). Which family the inner CV keeps selecting is the more honest
  version of that comparison than the grid maximum.
- n is 191 games; every mean travels with its fold std, and differences within
  ~1 std should not be narrated as real. The std across 5 folds x 3 seeds also
  *understates* uncertainty, because the 3 seeds reshuffle the same games -- for
  a claim that one family genuinely beats another, a game-level bootstrap would
  be the right test.
- Permutation importances are on held-out games; a feature at ~0 adds nothing
  out-of-sample even if its coefficient looks large in a full-data fit. They
  are reported for one configuration per LLM -- a feature ranking taken from a
  weaker configuration is not evidence about the LLM.
- L1 selection is unstable when features are correlated (accusation counts and
  utterance counts are): if two features carry the same signal, the path keeps
  one of them, and which one can flip across folds. `share_enters_first5` is
  the stability check.
- The circle covariates are *alternative-specific*: they are zero on every
  player row and only move the "No Werewolf" score. A feature that is constant
  within a game cannot change a within-game argmax, which is why game-level
  summaries had to enter this way rather than as ordinary columns.
- `top1_circle_votes` rests on few instances per fold. Read it as a check that
  the model is not simply ignoring the option, not as a precise rate.
